# Wrench control: DDPG and PID feedback

Training source selected because Comparision-All.ipynb loads the 1,000-episode checkpoint family.

Selected source excerpts from the original research notebook. Outputs, execution counts, widget state and duplicate plotting trials are removed. Definitions are kept as code; configuration, training launches and hardware calls are displayed as reference text so Run All cannot start training or connect to a robot. This is a review extract, not a complete reproducibility package. See README.md for omitted dependencies.


In [ ]:
import os
import random
from collections import deque
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
device = torch.device("cpu")


## Source cell 3 (original zero-based index 2)


Reference only; not executed.

```python
import os
import torch
import random
import numpy as np
from tqdm import tqdm
import torch.nn as nn
from utils import MatlabEnv_Static
from utils import MatlabEnv_Dynamic
import torch.optim as optim
from scipy.io import loadmat
from collections import deque
import matplotlib.pyplot as plt

# Configuration
current_dir = os.getcwd()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path configurations
PATHS = {
    'best_weights': os.path.join(current_dir, 'Weights', 'Best_weights-1000 eps'),
    'last_weights': os.path.join(current_dir, 'Weights', 'Last_weights-1000 eps'),
    'variables': os.path.join(current_dir, 'Variables-1000 eps'),
    'renders_train': os.path.join(current_dir, 'Renders', 'DRL-1000 eps','Train_DRL'),
    'render_tests': os.path.join(current_dir, 'Renders', 'DRL-1000 eps', 'Test_DRL'),
    'render_act_noise': os.path.join(current_dir, 'Renders', 'DRL-1000 eps', 'Action_noise_DRL'),
    'render_state_noise': os.path.join(current_dir, 'Renders', 'DRL-1000 eps', 'State_noise_DRL'),
    'plots_learning': os.path.join(current_dir, 'Plots', 'DRL-1000 eps','Learning_Curves'),
    'plots_test': os.path.join(current_dir, 'Plots', 'DRL-1000 eps','Tests'),
    'plots_act_noise': os.path.join(current_dir, 'Plots', 'DRL-1000 eps', 'Action_Noise'),
    'plots_state_noise': os.path.join(current_dir, 'Plots', 'DRL-1000 eps', 'State_Noise'),
    'plots_generated_points': os.path.join(current_dir, 'Plots', 'DRL-1000 eps', 'Generated_Points'),
    'matlab_workspaces': os.path.abspath(os.path.join(current_dir, '..', '..', '2) Matlab_Codes', '4) Workspaces', 'Cartesian'))
}

for path in PATHS.values():
    os.makedirs(path, exist_ok=True)
```


## Source cell 6 (original zero-based index 5)


Reference only; not executed.

```python
def get_dataset_paths():
    base_train = os.path.join(PATHS['matlab_workspaces'], 'PDMS2', 'Static', 'Spiral', 'Set2')
    base_test = os.path.join(PATHS['matlab_workspaces'], 'PDMS2', 'Static', 'Tests', 'Set2')

    return {
        'train': [
            os.path.join(base_train, f'Static_PDMS2_Spiral_Set2_{s}.mat')
            for s in ['0.30_0.10', '0.20_0.20', '0.22_0.18','0.15_0.16', '0.18_0.14', '0.25_0.14', '0.28_0.12', '0.12_0.11']
        ],
        'test': [
            os.path.join(base_test, f'Static_PDMS2_{t}_Set2_0.20_0.01.mat')
            for t in ['Spiral', 'Inf', 'Sine', 'Line', 'Square', 'Triangle']
        ]
    }

DATASETS = get_dataset_paths()
```


## Source cell 9 (original zero-based index 8)


In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, action_max, action_min, hidden_sizes, state_min, state_max):
        super().__init__()
        self.register_buffer('action_max', torch.tensor(action_max, dtype=torch.float32))
        self.register_buffer('action_min', torch.tensor(action_min, dtype=torch.float32))
        self.register_buffer('state_min', torch.tensor(state_min, dtype=torch.float32))
        self.register_buffer('state_max', torch.tensor(state_max, dtype=torch.float32))
        layers = []
        input_dim = state_dim  # State only (6D: force + torque)
        for size in hidden_sizes:
            layers.extend([nn.Linear(input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            input_dim = size
        layers.append(nn.Linear(input_dim, action_dim))
        self.net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
    
    def forward(self, state):
        state_norm = 2 * (state - self.state_min) / (self.state_max - self.state_min) - 1
        act_norm = torch.tanh(self.net(state_norm))
        # act_norm = torch.tanh(self.net(state))
        act = 0.5 * (act_norm + 1) * (self.action_max - self.action_min) + self.action_min
        return act


## Source cell 11 (original zero-based index 10)


In [ ]:
class Critic(nn.Module):
    def __init__(self, input_dim, hidden_sizes, input_min, input_max):
        super().__init__()
        self.register_buffer('input_min', torch.tensor(input_min, dtype=torch.float32))
        self.register_buffer('input_max', torch.tensor(input_max, dtype=torch.float32))
        layers = []
        for size in hidden_sizes:
            layers.extend([nn.Linear(input_dim, size), nn.LayerNorm(size), nn.ReLU()])
            input_dim = size
        layers.append(nn.Linear(input_dim, 1))
        self.q_net = nn.Sequential(*layers).to(device)
        self.apply(self._init_weights)
    
    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            nn.init.zeros_(m.bias)
    
    def forward(self, x):
        x_norm = 2 * (x - self.input_min) / (self.input_max - self.input_min) - 1
        return self.q_net(x_norm)
        # return self.q_net(x)


## Source cell 13 (original zero-based index 12)


In [ ]:
class ReplayBuffer:
    def __init__(self, state_dim, action_dim, capacity=int(1e6)):
        self.capacity = capacity
        self.idx = 0
        self.full = False
        
        self.states = torch.zeros((capacity, state_dim), device=device, dtype=torch.float32)
        self.actions = torch.zeros((capacity, action_dim), device=device, dtype=torch.float32)
        self.rewards = torch.zeros(capacity, device=device, dtype=torch.float32)
        self.next_states = torch.zeros((capacity, state_dim), device=device, dtype=torch.float32)
        self.dones = torch.zeros(capacity, device=device, dtype=torch.float32)
    
    def add(self, state, action, reward, next_state, done):
        idx = self.idx
        
        self.states[idx] = torch.as_tensor(state, device=device, dtype=torch.float32)
        self.actions[idx] = torch.as_tensor(action, device=device, dtype=torch.float32)
        self.rewards[idx] = torch.as_tensor(reward, device=device, dtype=torch.float32)
        self.next_states[idx] = torch.as_tensor(next_state, device=device, dtype=torch.float32)
        self.dones[idx] = torch.as_tensor(done, device=device, dtype=torch.float32)
        
        self.idx = (self.idx + 1) % self.capacity
        self.full = self.full or (self.idx == 0)
    
    def sample(self, batch_size):
        max_idx = self.capacity if self.full else self.idx
        indices = torch.randint(0, max_idx, (batch_size,), device=device)
        
        return (
            self.states[indices],
            self.actions[indices],
            self.rewards[indices],
            self.next_states[indices],
            self.dones[indices],
        )


## Source cell 19 (original zero-based index 18)


In [ ]:
class DDPG:
    def __init__(self, env, train_paths, show_full=False, actor_hidden=(256, 128, 64), critic_hidden=(256, 128, 64),
                 lr_actor=1e-4, lr_critic=5e-4, gamma=0.99, tau=0.005, batch_size=256,
                 noise_decay=0.95, noise_init=0.3, noise_min=0.1):
        self.env = env
        self.gamma = gamma
        self.tau = tau
        self.batch_size = batch_size
        self.noise = noise_init
        self.noise_decay = noise_decay
        self.noise_min = noise_min
        self.show_full = show_full
        self.video_dir = PATHS['renders_train']
        self.generated_points = []  # To collect random generated points for 3D plotting

        # Environment specs
        state_dim = env.observation_space.shape[0]  # 6 (force + torque)
        action_dim = env.action_space.shape[0]  # 6 (n0 + m0)
        
        # Actor input: state (6)
        actor_input_dim = state_dim
        
        # Critic input: state (6) + action (6) = 12
        critic_input_dim = state_dim + action_dim
        
        # Bounds
        low = self.env.observation_space.low
        high = self.env.observation_space.high
        
        state_min = low
        state_max = high
        
        act_low = self.env.action_space.low
        act_high = self.env.action_space.high
        
        critic_input_min = np.concatenate([state_min, act_low])
        critic_input_max = np.concatenate([state_max, act_high])
        
        # Networks
        self.actor = Actor(state_dim, action_dim, act_high, act_low, 
                           actor_hidden, state_min, state_max).to(device)
        self.actor_target = Actor(state_dim, action_dim, act_high, act_low, 
                                  actor_hidden, state_min, state_max).to(device)
        self.critic = Critic(critic_input_dim, critic_hidden, critic_input_min, critic_input_max).to(device)
        self.critic_target = Critic(critic_input_dim, critic_hidden, critic_input_min, critic_input_max).to(device)
        
        # Optimizers
        self.actor_optim = optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic_optim = optim.Adam(self.critic.parameters(), lr=lr_critic)
        
        # Replay buffer
        self.buffer = ReplayBuffer(state_dim, action_dim)
        
        # Sync targets
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.critic_target.load_state_dict(self.critic.state_dict())
        
        # Noise sigma based on action bounds
        self.action_sigma = np.minimum(np.abs(act_low) / 3, act_high / 3)

    def act(self, state, training=True):
        with torch.no_grad():
            state_t = torch.as_tensor(state, device=device, dtype=torch.float32)
            action = self.actor(state_t).cpu().numpy()
            if training:
                noise = self.noise * self.action_sigma * np.random.randn(*action.shape)
                # noise = self.noise * np.random.randn(*action.shape)
                action = np.clip(action + noise, self.env.action_space.low, self.env.action_space.high)
            return action
    
    def update(self):
        states, actions, rewards, next_states, dones = self.buffer.sample(self.batch_size)
        
        with torch.no_grad():
            target_actions = self.actor_target(next_states)
            critic_input_target = torch.cat([next_states, target_actions], dim=1)
            target_q = self.critic_target(critic_input_target).squeeze()
            target_q = rewards + (1 - dones) * self.gamma * target_q
        
        # Current Critic Q-value
        critic_input = torch.cat([states, actions], dim=1)
        current_q = self.critic(critic_input).squeeze()
        critic_loss = nn.MSELoss()(current_q, target_q)
        
        self.critic_optim.zero_grad()
        critic_loss.backward()
        # torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 1.0)
        self.critic_optim.step()
        
        # Actor loss
        actor_output = self.actor(states)
        critic_input_actor = torch.cat([states, actor_output], dim=1)
        actor_loss = -self.critic(critic_input_actor).mean()
        
        self.actor_optim.zero_grad()
        actor_loss.backward()
        # torch.nn.utils.clip_grad_norm_(self.actor.parameters(), 1.0)
        self.actor_optim.step()
        
        # Update targets
        with torch.no_grad():
            for t, s in zip(self.actor_target.parameters(), self.actor.parameters()):
                t.data.mul_(1 - self.tau).add_(s.data, alpha=self.tau)
            for t, s in zip(self.critic_target.parameters(), self.critic.parameters()):
                t.data.mul_(1 - self.tau).add_(s.data, alpha=self.tau)
        
        return critic_loss.item(), actor_loss.item()

    def train(self, episodes=200, steps=500):
        best_reward = float('-inf')
        returns, critic_losses, actor_losses, reach = [], [], [], []
        Errors, Sum_Errors, Diff_Errors, Act_Errors = [], [], [], []

        for ep in range(1, episodes+1):

            state = self.env.reset(new_ep=1)  # Random target force/torque
            self.generated_points.append(state.copy())  # Collect initial state
            ep_reward = 0
            ep_success, ep_error, ep_sum_e, ep_dif_e, ep_act_err = 0, [], [], [], []
            
            with tqdm(total=steps, desc=f'Episode {ep}/{episodes}', unit='step', miniters=10) as pbar:
                for step in range(steps):
                    action = self.act(state)
                    next_state, reward, done, terminated, success, error, sum_e, diff_e, act_error = self.env.step(action, state)  # Target is current state

                    ep_success += success
                    ep_error.append(error)
                    ep_sum_e.append(sum_e)
                    ep_dif_e.append(diff_e)
                    ep_act_err.append(act_error)


                    self.buffer.add(state, action, reward, next_state, done)
                    state = self.env.reset(new_ep=0)
                    self.generated_points.append(state.copy())  # Collect new state after reset
                    ep_reward += reward
                    
                    if self.buffer.idx > self.batch_size:
                        cl, al = self.update()
                        critic_losses.append(cl)
                        actor_losses.append(al)
                        if self.show_full:
                            pbar.set_postfix_str(f"Critic Loss: {cl:.3f} | Actor Loss: {al:.3f} | Score: {ep_reward:.3f} | Success: {ep_success/steps:.3f} | Error: {np.mean(ep_error):.3f} | Sum_E: {sum_e:.3f} | Diff_E: {np.mean(ep_dif_e):.3f} | Act_E: {np.mean(ep_act_err):.3f}")
                        else:
                            pbar.set_postfix_str(f"Critic Loss: {cl:.3f} | Actor Loss: {al:.3f} | Score: {ep_reward:.3f} | Success: {ep_success/steps:.3f}")
                    
                    pbar.update(1)

                    # if terminated:
                    #     break

            success_rate = ep_success / steps
            returns.append(ep_reward/steps)
            reach.append(success_rate)
            Errors.append(np.mean(ep_error))
            Sum_Errors.append(np.mean(ep_sum_e))
            Diff_Errors.append(np.mean(ep_dif_e))
            Act_Errors.append(np.mean(ep_act_err))

            # Save 3D plot of random points every 10 episodes
            if ep % 50 == 0 or ep == 5 or ep == 10 or ep == 20:
                save_path = os.path.join(PATHS['plots_generated_points'], f'random_points_up_to_ep_{ep}.png')
                plot_3d_random_points(self.generated_points, save_path, f'Cumulative Random Generated Points up to Episode {ep}')

            if ep%100 == 0:
                state = self.env.reset(new_ep=1)
                action = self.act(state, training=False)
                next_state, reward, done, terminated, success, error, sum_e, diff_e, act_error = self.env.step(action, state)  # Target is current state
                print(f"EP: {ep}|Rew: {reward:.3f} | Success: {success:.3f} | Error: {error:.3f}")
                print(f"S: {state}| NS: {next_state}")

            # Decay action noise every 10 episodes
            if ep % 10 == 0:
                self.noise = max(self.noise * self.noise_decay, self.noise_min)

            # Save model checkpoints
            torch.save(self.actor.state_dict(), os.path.join(PATHS['last_weights'], 'actor_dict.pth'))
            torch.save(self.critic.state_dict(), os.path.join(PATHS['last_weights'], 'critic_dict.pth'))
            with open(os.path.join(PATHS['last_weights'], "last_reward.txt"), 'w') as f:
                f.write(f"Last Episode: {ep+1}\nReward: {ep_reward}\n")
            
            if ep_reward >= best_reward:
                best_reward = ep_reward
                torch.save(self.actor.state_dict(), os.path.join(PATHS['best_weights'], 'actor_dict.pth'))
                torch.save(self.critic.state_dict(), os.path.join(PATHS['best_weights'], 'critic_dict.pth'))
                with open(os.path.join(PATHS['best_weights'], "best_reward.txt"), 'w') as f:
                    f.write(f"Best Episode: {ep+1}\nReward: {ep_reward}\n")

            # Render video every 30 episodes
            if ep % 50 == 0 or ep==20:
                print(f"Rendering video for episode {ep}...")
                render_file = DATASETS['train'][0] if DATASETS['train'] else None
                video_path_main = os.path.join(self.video_dir, f'episode_{ep}_render_main.mp4')
                video_path_setup = os.path.join(self.video_dir, f'episode_{ep}_render_setup.mp4')
                render_movement(self, self.env, render_file, video_path_main, video_path_setup, show=False)

        return returns, reach, Errors, Sum_Errors, Diff_Errors, Act_Errors, critic_losses, actor_losses, self.actor, self.critic


## Source cell 21 (original zero-based index 20)


Reference only; not executed.

```python
matter = 'PDMS2'
mag_set = 'Set2'
env = MatlabEnv_Static(matter=matter, mag_set=mag_set)  # Assumes revised env
```


## Source cell 23 (original zero-based index 22)


Reference only; not executed.

```python
%%time

agent = DDPG(
    env,
    train_paths=DATASETS['train'],
    show_full=True,
    actor_hidden=(400, 300),
    critic_hidden=(400, 300),
    lr_actor=1e-4,
    lr_critic=2e-4,
    batch_size=256,
    gamma=0.99,
    tau=0.005,
    noise_decay=0.99,
    noise_init=0.5,
    noise_min=0.01
)

returns, success_rate, errors, sum_errors, diff_errors, act_errors, critic_losses, actor_losses, actor_model, critic_model = agent.train(episodes=int(1000), steps=499)

env.close()
```
